# ساهر × ComfyUI على Google Colab

دفتر تجربة تفاعلي: يشغّل ComfyUI محلياً داخل الجلسة، ويرسل وصف الفيديو من خلية الدفتر عبر محول Saher، ثم يعرض MP4 هنا. لا حاجة إلى Tunnel أو رابط عام أو واجهة ويب خارج دفتر Colab.

> **تنبيه سياسة وموارد:** توضح Google أن موارد Colab المجانية غير مضمونة، وأن حصص GPU ونوعه والمدة تتغير. كما تحذر من تجاوز واجهة الدفتر للتفاعل أساساً عبر واجهة ويب لتوليد المحتوى. لهذا يستخدم هذا الدفتر واجهة Colab نفسها فقط، ويترك واجهة Saher/ComfyUI الخارجية غير مكشوفة. لا تستخدمه لإنشاء deepfakes أو محتوى يخالف سياسة Google أو رخصة النموذج. [سياسة Colab](https://research.google.com/colaboratory/faq.html)

يتطلب التوليد GPU NVIDIA بذاكرة مناسبة. دليل ComfyUI يذكر أن Wan2.1 T2V-1.3B يحتاج نحو 8 GB VRAM على الأقل؛ ذلك إرشاد رسمي وليس ضماناً لأي جلسة. سنستخدم 480p و17 إطاراً لفحص قصير، وقد يظل التنزيل أو التوليد بطيئاً أو يفشل.

**طريقة الاستخدام:** اختر `Runtime → Change runtime type → T4 GPU` إن كان متاحاً، ثم شغّل الخلايا بالترتيب. لا تفعّل Google Drive ولا ترفع أي مفاتيح. تُحذف الملفات عند انتهاء الجلسة.

## 1. فحص Python وGPU قبل التنزيل

In [ ]:
import sys, torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('لم يتوفر GPU في هذه الجلسة. اختر Runtime > Change runtime type > GPU، أو حاول لاحقاً؛ التوفر المجاني غير مضمون.')
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
print('GPU:', props.name, '| VRAM GiB:', round(vram_gib, 1))
if vram_gib < 8:
    raise RuntimeError(f'الذاكرة المعروضة {vram_gib:.1f} GiB أقل من حد ComfyUI الموصى به لهذا المسار (نحو 8 GiB). أوقف التجربة لتجنب تنزيل الأوزان ثم الفشل.')
print('GPU check passed. This is only a hardware check, not a generation benchmark.')

## 2. تنزيل ComfyUI وفرع Saher التجريبي وتثبيت الاعتماديات

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git
!git clone --depth 1 --branch feature/saher-mvp-foundation https://github.com/jonbander/saher-video-colab.git
%pip install -r /content/ComfyUI/requirements.txt
%pip install -e /content/saher-video-colab

## 3. تنزيل أوزان Wan2.1 الرسمية إلى مساحة الجلسة

تُنزّل الخلية نموذج T2V-1.3B بنسخة fp16، ومشفّر النص fp8، وVAE من مستودع Comfy-Org الرسمي. مجموع الأوزان كبير وقد يتطلب عدة غيغابايت من التنزيل والمساحة؛ لا تحفظها في Drive تلقائياً. يوصي مؤلف Wan2.1 بدقة 480p لهذا الطراز. [دليل ComfyUI الرسمي](https://docs.comfy.org/tutorials/video/wan/wan-video)

In [ ]:
from pathlib import Path
import shutil
from huggingface_hub import HfApi, hf_hub_download

COMFY = Path('/content/ComfyUI')
MODEL_REPO = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'
revision = HfApi().model_info(MODEL_REPO).sha
print('Pinned model repository revision for this run:', revision)

files = {
    'split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors': COMFY/'models/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors',
    'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors': COMFY/'models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    'split_files/vae/wan_2.1_vae.safetensors': COMFY/'models/vae/wan_2.1_vae.safetensors',
}
for repo_file, target in files.items():
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f'Downloading {Path(repo_file).name} ...')
        cached = Path(hf_hub_download(repo_id=MODEL_REPO, filename=repo_file, revision=revision, local_dir=str(COMFY/'models')))
        shutil.move(cached, target)
    print(f'{target.name}: {target.stat().st_size/1024**3:.2f} GiB')
print('Model files ready.')

## 4. إنشاء workflow API صغير قابل للربط بمحول Saher

يعادل هذا الرسم workflow Wan2.1 T2V الرسمي في ComfyUI: تحميل نموذج 1.3B، وترميز الوصف، وإنشاء latent بحجم 832×480 و17 إطاراً، ثم حفظ الفيديو. لا نفتح واجهة ComfyUI في المتصفح.

In [ ]:
import json

workflow = {
  '37': {'class_type':'UNETLoader','inputs':{'unet_name':'wan2.1_t2v_1.3B_fp16.safetensors','weight_dtype':'default'}},
  '38': {'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','type':'wan','device':'default'}},
  '39': {'class_type':'VAELoader','inputs':{'vae_name':'wan_2.1_vae.safetensors'}},
  '6': {'class_type':'CLIPTextEncode','inputs':{'clip':['38',0],'text':'A cinematic shot of a small boat crossing a calm lake at sunrise, slow camera movement, soft golden light'}},
  '7': {'class_type':'CLIPTextEncode','inputs':{'clip':['38',0],'text':'blurry, distorted, low quality, text, watermark'}},
  '40': {'class_type':'EmptyHunyuanLatentVideo','inputs':{'width':832,'height':480,'length':17,'batch_size':1}},
  '48': {'class_type':'ModelSamplingSD3','inputs':{'model':['37',0],'shift':8}},
  '3': {'class_type':'KSampler','inputs':{'model':['48',0],'positive':['6',0],'negative':['7',0],'latent_image':['40',0],'seed':20261025,'steps':20,'cfg':6,'sampler_name':'uni_pc','scheduler':'simple','denoise':1.0}},
  '8': {'class_type':'VAEDecode','inputs':{'samples':['3',0],'vae':['39',0]}},
  '49': {'class_type':'CreateVideo','inputs':{'images':['8',0],'fps':16}},
  '50': {'class_type':'SaveVideo','inputs':{'video':['49',0],'filename_prefix':'video/SaherTest','format':'auto','codec':'auto'}}
}
workflow_path = Path('/content/saher-wan21-api.json')
workflow_path.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved API workflow:', workflow_path)

## 5. تشغيل ComfyUI محلياً داخل الجلسة فقط

In [ ]:
import subprocess, time, requests
from pathlib import Path

log_file = open('/content/comfyui.log', 'w')
comfy_process = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'], cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT)
base = 'http://127.0.0.1:8188'
for attempt in range(120):
    if comfy_process.poll() is not None:
        raise RuntimeError('ComfyUI exited early. Inspect /content/comfyui.log. Last log lines:\n' + Path('/content/comfyui.log').read_text(errors='replace')[-5000:])
    try:
        health = requests.get(base + '/system_stats', timeout=3)
        if health.ok:
            print('ComfyUI local API ready on loopback; it is not exposed through a public tunnel.')
            print('Reported device:', health.json().get('system',{}).get('pytorch_version','PyTorch'))
            break
    except requests.RequestException:
        time.sleep(2)
else:
    raise RuntimeError('ComfyUI startup timed out. Inspect /content/comfyui.log.')

## 6. استدعاء API الخاص بساهر وإظهار تقدم المهمة

تستخدم الخلية واجهة TestClient داخلياً في عملية الدفتر نفسها، وترسل المهمة من notebook إلى تطبيق Saher ثم إلى ComfyUI. لا تُشغّل خادماً عاماً، ولا تتجاوز واجهة Colab. الزمن متغير، وقد يوقف Colab الجلسة قبل اكتمال المهمة.

In [ ]:
import os, time
from fastapi.testclient import TestClient

os.environ['COMFYUI_BASE_URL'] = 'http://127.0.0.1:8188'
os.environ['COMFYUI_WORKFLOW_PATH'] = str(workflow_path)
os.environ['COMFYUI_PROMPT_NODE_ID'] = '6'
os.environ['COMFYUI_PROMPT_INPUT'] = 'text'

# Ensure a fresh module reads the workflow path set above.
import importlib, saher_video.app
importlib.reload(saher_video.app)
client = TestClient(saher_video.app.app)
prompt = input('اكتب وصف المشهد بالإنجليزية (أو اضغط Enter لاستخدام الوصف التجريبي): ').strip()
if prompt:
    # Update the positive-prompt node in the owned local API workflow, not a user-supplied file.
    workflow['6']['inputs']['text'] = prompt
    workflow_path.write_text(json.dumps(workflow, ensure_ascii=False, indent=2), encoding='utf-8')
else:
    prompt = workflow['6']['inputs']['text']

submitted = client.post('/api/v1/jobs', json={'prompt': prompt})
assert submitted.status_code == 202, submitted.text
job_id = submitted.json()['job_id']
print('Saher job id:', job_id)

job = None
for attempt in range(180):  # Up to 30 minutes; Colab may terminate sooner.
    response = client.get(f'/api/v1/jobs/{job_id}')
    if response.status_code != 200:
        raise RuntimeError(response.text)
    job = response.json()
    if attempt % 3 == 0:
        print(f"Elapsed ~{attempt*10}s | state={job['state']}")
    if job['state'] == 'completed':
        break
    if job['state'] == 'failed':
        raise RuntimeError(job.get('error') or 'ComfyUI generation failed')
    if job['state'] == 'unknown':
        raise RuntimeError('ComfyUI no longer has this job in its queue/history')
    time.sleep(10)
else:
    raise TimeoutError('Stopped waiting after 30 minutes. The Colab session may still be running; inspect comfyui.log.')

videos = [item for item in job['outputs'] if item['content_type'].startswith('video/')]
if not videos:
    raise RuntimeError(f'No video output returned. Check that the workflow has a SaveVideo node. Outputs: {job["outputs"]}')
asset = client.get(videos[0]['url'])
assert asset.ok, asset.text[:1000]
output_path = Path('/content/saher-colab-test.mp4')
output_path.write_bytes(asset.content)
print(f'Video saved temporarily: {output_path} ({output_path.stat().st_size/1024**2:.1f} MiB)')
from IPython.display import Video, display
display(Video(str(output_path), embed=True, width=480))

## 7. إيقاف المحرك

بعد حفظ/تنزيل الفيديو من مخرجات الدفتر، شغّل الخلية لإيقاف ComfyUI. ملفات `/content` مؤقتة، ولا يُحفظ الفيديو في Google Drive تلقائياً.

In [ ]:
if 'comfy_process' in globals() and comfy_process.poll() is None:
    comfy_process.terminate()
    try:
        comfy_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        comfy_process.kill()
    print('ComfyUI stopped.')
else:
    print('ComfyUI is not running.')

### المصادر

- [أسئلة Google Colab وحدود الموارد والأنشطة](https://research.google.com/colaboratory/faq.html)
- [ComfyUI: Wan2.1 T2V-1.3B workflow وملفات النموذج](https://docs.comfy.org/tutorials/video/wan/wan-video)
- [قالب Wan2.1 الرسمي في ComfyUI](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/text_to_video_wan.json)
- [بطاقة Wan2.1 T2V-1.3B ورخصة Apache-2.0 وشروط الاستخدام](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B)
- [Saher — المستودع](https://github.com/jonbander/saher-video-colab/tree/feature/saher-mvp-foundation)